[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C43_Data_Engineering_Course/02_streaming_loaders/02_streaming_loaders.ipynb)

# 02 · 流式加载与分片（用 numpy/标准库从零模拟 + 对拍 + 算账）

目标：把 **流式分片 → shard 分配(不重不漏) → shuffle 缓冲区近似全局打乱 → 预取流水 → 吞吐建模** 从零实现，用**指标量化**打乱质量、用**吞吐账**定位瓶颈。

路线：流式 vs 全量内存 → shard 分配不变量 → shuffle 缓冲模拟 → 打乱质量(位移+相关性) vs 缓冲大小 → 预取流水重叠 → 吞吐瓶颈 → ✏️ 练习 → 📖 答案 → 🧪 真实加载器胶囊。

> 心智模型：**数据装不下内存 -> 一切都是「有限内存的近似」：流式≈随机访问、缓冲区≈全局打乱、预取≈零延迟**。我们把每个近似的*质量*量化出来。

## 1 · 流式 vs 全量入内存：内存与 D 解耦

全量打乱要把整个数据集装进内存（内存 ∝ 数据量 D）；流式只持有一个**固定大小**的工作集（内存与 D 无关）。
先用一个「生成器流」模拟流式读取，确认它**逐条产出、不缓存全部**。

In [ ]:
import numpy as np
from collections import deque
rng = np.random.default_rng(0)

def stream_samples(n):
    '''流式数据源：逐条 yield，绝不一次性建全量列表（模拟从 shard 顺序读）。'''
    for i in range(n):
        yield i                      # 这里用样本下标代表一条样本

# 流式消费：任意时刻只持有当前一条
seen, peak_held = 0, 1
for x in stream_samples(1_000_000):
    seen += 1                        # 处理一条（这里只计数）
assert seen == 1_000_000
# 对照：全量入内存才能做的事（这里只示意它需要 O(D) 空间）
small = list(stream_samples(1000))   # 全量列表：内存 ∝ 条数
assert len(small) == 1000
print(f'流式消费了 100 万条，任意时刻只持有 ~{peak_held} 条（内存与 D 无关）')
print(f'全量入内存 1000 条需要一个长度 {len(small)} 的列表（内存 ∝ D，50TB 时不可行）')
print('✅ 流式让内存与数据量解耦 —— PB 级加载的前提')

## 2 · Shard 分配：覆盖 + 互斥两条不变量

把 `S` 个 shard 分给 `R` 个 rank。必须满足：**覆盖**（所有 shard 都被分配）+ **互斥**（无 shard 被分给两个 rank）。
用取模分配，并把两条不变量写成 `assert`——它们一旦破坏，训练会**静默地用错数据**。

In [ ]:
def assign_shards(shard_ids, rank, world_size):
    '''纯函数分配：第 i 个 shard 给 i % world_size。返回该 rank 分到的 shard 列表。'''
    return [s for k, s in enumerate(shard_ids) if k % world_size == rank]

S, R = 103, 8                       # 故意不整除
shard_ids = list(range(S))
assignments = [assign_shards(shard_ids, r, R) for r in range(R)]

# 不变量①覆盖：并集 == 全部
union = set().union(*assignments)
assert union == set(shard_ids), '覆盖被破坏：有 shard 没人读！'
# 不变量②互斥：两两交集为空
for i in range(R):
    for j in range(i+1, R):
        assert not (set(assignments[i]) & set(assignments[j])), '互斥被破坏：shard 被重复读！'

loads = [len(a) for a in assignments]
print(f'{S} 个 shard 分给 {R} 个 rank -> 每 rank 分到 {min(loads)}~{max(loads)} 个')
print(f'负载不均：最多与最少差 {max(loads)-min(loads)} 个 shard（straggler 风险）')
assert max(loads) - min(loads) <= 1, '取模分配的负载差应 <=1 个 shard'
print('✅ 覆盖 + 互斥成立；不整除导致 ±1 的负载差 —— 这是 straggler 的根源')

## 3 · 陷阱：shard 数 < rank 数

若 shard 数少于 rank 数，部分 rank **分不到任何 shard**、空转甚至死锁在梯度同步。
验证这个陷阱，并给出工程规则：**shard 数应远大于 rank 数**。

In [ ]:
def starved_ranks(S, R):
    '''返回分不到 shard 的 rank 数。'''
    shard_ids = list(range(S))
    empty = sum(1 for r in range(R) if len(assign_shards(shard_ids, r, R)) == 0)
    return empty

print(f"{'shard 数 S':>10s} {'rank 数 R':>10s} {'空转 rank':>10s}")
for S, R in [(800, 1000), (1000, 1000), (4000, 1000)]:
    print(f'{S:>10d} {R:>10d} {starved_ranks(S, R):>10d}')
# S<R 时必有 rank 空转
assert starved_ranks(800, 1000) == 200, 'S=800,R=1000 应有 200 个 rank 空转'
assert starved_ranks(4000, 1000) == 0, 'S>=R 且整除时无人空转'
# 经验规则：shard 数 >= 数倍 rank 数
RULE = 4
assert 4000 >= RULE * 1000
print('\n✅ 规则：shard 数应 >= 数倍 rank 数，否则 GPU 空转 —— 分片粒度与并行规模绑死')

## 4 · Shuffle 缓冲区模拟器（核心）

缓冲区容量 `B`：先填满，再「随机弹一个吐出、补一个进来」。
`B=1` = 不打乱；`B=n` = 真·全局打乱。先实现它，验证输出是输入的一个**排列**（不丢不重）。

In [ ]:
def shuffle_buffer(stream, B, seed=0):
    '''流式近似打乱：维护容量 B 的缓冲，每步随机弹出一个、再补入下一个流入样本。'''
    r = np.random.default_rng(seed)
    buf = []
    it = iter(stream)
    # 先填满缓冲
    for x in it:
        buf.append(x)
        if len(buf) >= B:
            break
    # 稳态：弹一个、补一个
    for x in it:
        j = r.integers(0, len(buf))
        out = buf[j]; buf[j] = x          # 随机弹出 out，用新流入 x 顶替
        yield out
    # 收尾：缓冲里剩的随机吐完
    r.shuffle(buf)
    for x in buf:
        yield x

n = 1000
out = list(shuffle_buffer(range(n), B=50, seed=1))
# 正确性：输出必须是输入的一个排列（流式打乱不丢不重）
assert sorted(out) == list(range(n)), '打乱后必须是原集合的排列（不丢不重）'
assert len(out) == n
# B=1 等于不打乱
assert list(shuffle_buffer(range(n), B=1, seed=1)) == list(range(n))
print(f'B=50 打乱 {n} 条：输出是输入的排列 ✅（不丢不重）')
print('B=1 时输出==输入（完全不打乱）✅')
print('✅ shuffle 缓冲器正确：内存只占 B 条，输出被局部打乱')

## 5 · 打乱质量 vs 缓冲大小：两个指标

用**平均位移**（每条样本输入→输出移动的距离均值）量化打乱强度。
- 真随机打乱期望位移 ≈ n/3；不打乱位移 = 0。
- 规律：位移随 B 增大而增大，但**接近 n 时饱和**。这告诉你 buffer_size 该设多大。

In [ ]:
def avg_displacement(perm):
    '''perm[k] = 输出第 k 位上的原始下标。返回平均 |输出位置 - 输入位置|。'''
    perm = np.asarray(perm)
    in_pos = perm                       # 原始下标 = 输入位置
    out_pos = np.arange(len(perm))       # 输出位置
    return float(np.mean(np.abs(out_pos - in_pos)))

n = 2000
print(f"{'缓冲 B':>8s} {'平均位移':>10s} {'占 n/3 比例':>12s}")
rand_ref = avg_displacement(np.random.default_rng(9).permutation(n))
disp = {}
for B in [1, 10, 50, 200, 1000, n]:
    perm = list(shuffle_buffer(range(n), B=B, seed=2))
    d = avg_displacement(perm)
    disp[B] = d
    print(f'{B:>8d} {d:>10.1f} {d/(n/3):>11.0%}')
print(f'(真随机打乱参考位移 ≈ {rand_ref:.0f} ≈ n/3 = {n/3:.0f})')
# 单调性 + 饱和：B 越大位移越大；B=n 接近真随机
assert disp[1] == 0.0, 'B=1 不打乱，位移=0'
assert disp[10] < disp[200] < disp[1000], '位移随 B 单调增'
assert disp[n] > 0.8 * rand_ref, 'B=n 应接近真随机打乱的位移'
print('✅ 位移随 B 增大而增、接近 n 时饱和 —— B 取「拐点」即用小内存拿到近全局打乱')

位移大不等于「相似邻居被拆散」。若**相邻样本天然相似**（同源连续网页），
缓冲区可能位移很大却没拆散相似邻居。用「输出里仍来自原始邻近窗口的比例」近似这个相关性。

In [ ]:
def neighbor_locality(perm, window=5):
    '''输出相邻两条，其原始下标之差 <= window 的比例。越高=越多老邻居还挨着。'''
    perm = np.asarray(perm)
    close = np.abs(np.diff(perm)) <= window
    return float(np.mean(close))

n = 2000
print(f"{'缓冲 B':>8s} {'老邻居比例':>12s}")
for B in [1, 50, 500]:
    perm = list(shuffle_buffer(range(n), B=B, seed=3))
    loc = neighbor_locality(perm)
    print(f'{B:>8d} {loc:>11.1%}')
# B 越大，老邻居越少（相似邻居越可能被拆散）
loc1 = neighbor_locality(list(shuffle_buffer(range(n), B=1, seed=3)))
loc500 = neighbor_locality(list(shuffle_buffer(range(n), B=500, seed=3)))
assert loc1 == 1.0, 'B=1 全是老邻居（完全没打乱）'
assert loc500 < 0.1, 'B=500 几乎拆散所有老邻居'
print('✅ 位移 + 邻居相关性两个指标一起看：只调 buffer 不够，还要管 shard 顺序与写入 interleave')

## 6 · 预取流水重叠：吞吐 = 1 / max(各级)

加载分多级（读IO/解压/token/计算）。串行总时间 = 各级之和；**流水重叠**后稳态吞吐 = 1/max(各级)。
建一个流水模型，对拍「串行」与「重叠」的总时间，并验证瓶颈级决定吞吐。

In [ ]:
def serial_time(stage_times, n_batches):
    '''串行：每个 batch 顺序走完所有级。'''
    return n_batches * sum(stage_times)

def pipelined_time(stage_times, n_batches):
    '''流水重叠：填满流水后，每个 batch 间隔 = 最慢级；总 = 填充 + (n-1)*瓶颈。'''
    bottleneck = max(stage_times)
    fill = sum(stage_times)               # 第一个 batch 走完全程（填充流水）
    return fill + (n_batches - 1) * bottleneck

stages = [4.0, 2.0, 3.0, 5.0]            # 读IO/解压/token/计算 每 batch 耗时
n_batches = 100
ts = serial_time(stages, n_batches)
tp = pipelined_time(stages, n_batches)
print(f'串行总时间   = {ts:.0f}（各级之和 × {n_batches}）')
print(f'流水重叠总时间 = {tp:.0f}（≈ 瓶颈级 {max(stages)} × {n_batches}）')
print(f'加速 ≈ {ts/tp:.2f}x；稳态吞吐 = 1/{max(stages)} = {1/max(stages):.3f} batch/单位时间')
assert tp < ts, '流水重叠应快于串行'
# 稳态由瓶颈级支配：把非瓶颈级(解压2.0)再加速不改变稳态吞吐
stages_faster_nonbottleneck = [4.0, 0.5, 3.0, 5.0]
assert pipelined_time(stages_faster_nonbottleneck, n_batches) == tp - (2.0-0.5), '非瓶颈级提速只省了填充'
# 只有降瓶颈级才显著提稳态
stages_lower_bottleneck = [4.0, 2.0, 3.0, 3.5]
assert pipelined_time(stages_lower_bottleneck, n_batches) < tp, '降低瓶颈级才真正提吞吐'
print('✅ 吞吐由瓶颈级决定：优化非瓶颈级几乎无效 —— 先定位瓶颈再动手（Amdahl 日常版）')

---
## ✏️ 练习 1：带 epoch 种子的 shard 分配 + 打乱

固定分配会让每个 epoch 看到相同顺序。实现 `assign_and_shuffle(shard_ids, rank, world_size, epoch, seed=0)`：
先用 `(seed, epoch)` **确定性地打乱** shard 全序，再取模分给该 rank，返回该 rank 的 shard 列表。

要求：① 仍满足覆盖+互斥；② 同一 (epoch) 下确定可复现；③ 不同 epoch 顺序不同。

In [ ]:
def assign_and_shuffle(shard_ids, rank, world_size, epoch, seed=0):
    # TODO: 用 np.random.default_rng((seed, epoch)) 打乱 shard_ids 的副本（得到全序），
    #       再取打乱后序列里 index % world_size == rank 的那些，返回列表
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
S, R = 100, 8
ids = list(range(S))
e0 = [assign_and_shuffle(ids, r, R, epoch=0, seed=42) for r in range(R)]
# 覆盖 + 互斥
assert set().union(*e0) == set(ids), '覆盖被破坏'
for i in range(R):
    for j in range(i+1, R):
        assert not (set(e0[i]) & set(e0[j])), '互斥被破坏'
# 确定性可复现
assert assign_and_shuffle(ids, 0, R, epoch=0, seed=42) == assign_and_shuffle(ids, 0, R, epoch=0, seed=42)
# 不同 epoch 顺序应不同（rank 0 拿到的 shard 集合大概率变化）
e1_r0 = assign_and_shuffle(ids, 0, R, epoch=1, seed=42)
assert e1_r0 != e0[0], '不同 epoch 应打乱出不同分配'
print('✅ 练习 1 通过：带 epoch 种子的分配，可复现且每 epoch 不同，仍不重不漏')

## ✏️ 练习 2：shuffle 缓冲区的偏差——首条样本能跑多远？

缓冲区有个隐藏偏差：**最早流入的样本最晚才有机会被推迟**。实现 `max_delay_of_first(n, B, seed, trials)`：
对 `trials` 次随机，统计「输入第 0 条样本」的输出位置，返回其**最大值**（它最远能被推到第几位）。

直觉验证：第 0 条最多被推迟到约 O(B) 附近的位置之外不太可能很远（它一旦被弹出就走了）。

In [ ]:
def max_delay_of_first(n, B, seed=0, trials=200):
    # TODO: 跑 trials 次 shuffle_buffer(range(n), B, seed=...)，
    #       每次记录元素 0 出现在输出里的位置(index)，返回这些位置的最大值
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
n, B = 500, 20
md0 = max_delay_of_first(n, B, seed=0, trials=300)
# 第 0 条的输出位置应该是个合理范围内的整数
assert 0 <= md0 < n
# 缓冲越大，首条能被推得越远（视野 O(B)）
md_small = max_delay_of_first(n, 5, seed=0, trials=300)
md_large = max_delay_of_first(n, 100, seed=0, trials=300)
assert md_large > md_small, '更大的缓冲让首条能被推迟得更远'
print(f'B=5 时首条最远到位置 {md_small}；B=100 时最远到 {md_large}')
print('✅ 练习 2 通过：缓冲区打乱视野 ≈ O(B)，首条样本的推迟范围随 B 增大')

## ✏️ 练习 3：吞吐建模——加几个 worker 才不饿？

数据侧单 worker 准备一个 batch 要 `t_prep` 秒，GPU 算一个 batch 要 `t_compute` 秒。
`W` 个 worker 并行准备，则数据侧每 batch 的有效间隔 ≈ `t_prep / W`。

实现 `min_workers(t_prep, t_compute)`：返回让 GPU **不饿**（数据侧间隔 ≤ 计算）所需的最小 worker 数。

In [ ]:
def min_workers(t_prep, t_compute):
    # TODO: 找最小整数 W 使 t_prep / W <= t_compute，即 W >= t_prep / t_compute
    #       用 math.ceil；注意 W 至少为 1
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
import math
# 准备比计算慢 6.5 倍 -> 需要 7 个 worker
assert min_workers(t_prep=6.5, t_compute=1.0) == 7
# 准备本就比计算快 -> 1 个 worker 够
assert min_workers(t_prep=0.5, t_compute=1.0) == 1
# 刚好相等 -> 1 个
assert min_workers(t_prep=1.0, t_compute=1.0) == 1
# 验证：用返回的 W，数据侧间隔确实 <= 计算
W = min_workers(6.5, 1.0)
assert 6.5 / W <= 1.0
print(f'准备慢 6.5x -> 需 {min_workers(6.5,1.0)} 个 worker 才能喂饱 GPU')
print('✅ 练习 3 通过：能据准备/计算耗时算出防饥饿所需的 worker 数')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def assign_and_shuffle(shard_ids, rank, world_size, epoch, seed=0):
    order = list(shard_ids)
    np.random.default_rng((seed, epoch)).shuffle(order)   # 确定性全序打乱
    return [s for k, s in enumerate(order) if k % world_size == rank]

In [ ]:
# 练习 2 参考答案
def max_delay_of_first(n, B, seed=0, trials=200):
    worst = 0
    for t in range(trials):
        out = list(shuffle_buffer(range(n), B=B, seed=seed + t))
        pos = out.index(0)                # 元素 0 在输出里的位置
        worst = max(worst, pos)
    return worst

In [ ]:
# 练习 3 参考答案
def min_workers(t_prep, t_compute):
    import math
    return max(1, math.ceil(t_prep / t_compute))

---
## 🧪 真实数据胶囊：50 TB 语料的流式加载算一笔账

用真实加载器的公开配置（WebDataset / Mosaic StreamingDataset / HF streaming）算账：
① shard 切多大、切多少个；② shuffle 缓冲该设多大；③ 要多少聚合吞吐喂饱一个 GPU 集群。

（带 try/except：本环境不联网，直接用内置真实量级数字。）

In [ ]:
# 真实量级（约数：FineWeb/Dolma 级语料 + H100 集群）
DATASET_TB = 50.0           # 去重后语料 ~50 TB
SHARD_MB = 512              # WebDataset/MDS 典型 shard 大小
N_GPUS = 256                # 训练集群规模
PER_GPU_MBPS = 400.0        # 每 GPU 每秒要吃的数据(MB)，量级

# ① shard 数量
n_shards = DATASET_TB * 1e6 / SHARD_MB
print(f'① {DATASET_TB} TB 切成 {SHARD_MB} MB 的 shard -> {n_shards:,.0f} 个 shard')
print(f'   shard 数 {n_shards:,.0f} ≫ GPU 数 {N_GPUS}（满足「远大于」规则 ✅）')

# ② shuffle 缓冲：取「位移拐点」量级，常 ~1e4 条；内存 = B × 每条字节
B = 10_000
bytes_per_sample = 4 * 1024          # 每条 ~4KB(token 序列)
buf_mem_mb = B * bytes_per_sample / 1e6
print(f'② shuffle 缓冲 B={B:,} 条 -> 每 worker 占内存 ~{buf_mem_mb:.0f} MB（与 50TB 无关，O(B)）')

# ③ 聚合吞吐
agg_gbps = N_GPUS * PER_GPU_MBPS / 1000
print(f'③ 喂饱 {N_GPUS} 个 GPU 需聚合吞吐 ~{agg_gbps:.0f} GB/s -> 靠成百上千 worker 并行顺序拉 shard')

assert n_shards > N_GPUS * 4, 'shard 数应远大于 GPU 数'
assert buf_mem_mb < DATASET_TB * 1e6, 'shuffle 缓冲内存应远小于数据总量（这正是流式的意义）'
print('\n账目结论：流式让内存=O(B) 与 50TB 解耦；shard 数远超 GPU 数保证并行；')
print('聚合吞吐靠大量 worker 顺序拉 shard 凑齐 —— 加载器决定集群利用率。')

**🧪 胶囊练习**：实现 `shuffle_buffer_mem(B, bytes_per_sample, n_workers)`：估算**所有 worker 的 shuffle 缓冲总内存（字节）** = `B × bytes_per_sample × n_workers`。返回字节数。（用它判断给定内存预算下 B 能开多大。）

In [ ]:
def shuffle_buffer_mem(B, bytes_per_sample, n_workers):
    # TODO: 返回 B * bytes_per_sample * n_workers
    raise NotImplementedError

In [ ]:
# 自测
mem = shuffle_buffer_mem(B=10_000, bytes_per_sample=4096, n_workers=64)
assert mem == 10_000 * 4096 * 64
# 关键：总内存与数据集大小 D 无关，只与 (B, worker 数) 有关
assert mem < 50e12, '缓冲总内存应远小于 50TB 数据集'
print(f'B=1e4, 4KB/条, 64 worker -> shuffle 缓冲总内存 {mem/1e9:.2f} GB（与 50TB 数据无关）')
print('✅ 胶囊练习通过：shuffle 缓冲内存是 O(B × worker)，与数据量解耦')

In [ ]:
# 📖 胶囊参考答案
def shuffle_buffer_mem(B, bytes_per_sample, n_workers):
    return B * bytes_per_sample * n_workers

---
## 🔧 旁注：真实加载器里这些机制叫什么

本课的小模拟，在真实加载器里对应：

- **shuffle 缓冲**：HuggingFace `IterableDataset.shuffle(buffer_size=10000)` 的 `buffer_size` 就是本课的 `B`；跑完本课你确切知道它该设多大（位移拐点）。
- **shard 格式**：WebDataset 把样本打包成 `.tar`、顺序读；Mosaic StreamingDataset 用 MDS 格式 + 索引支持确定性恢复。
- **shard 分配 + epoch 打乱**：PyTorch `DistributedSampler(shuffle=True)` 按 rank 分样本并带 epoch 种子重洗；Mosaic 的 py1b/py1s 是更省内存的全局打乱算法。
- **预取**：PyTorch DataLoader 的 `num_workers` + `prefetch_factor`；NVIDIA DALI 把解码也搬上 GPU 流水。

你在 numpy 里验证过的分配不变量、缓冲打乱、吞吐瓶颈逻辑，直接对应这些库的参数与行为。

### 小结
- 数据装不进内存 -> **流式**：内存与数据量 D 解耦。
- **shard** 是并行/恢复/IO 的甜点（100MB~1GB，数量远超 rank 数）；分配要满足**覆盖+互斥**且带 epoch 种子。
- **shuffle 缓冲**用 O(B) 内存近似全局打乱；打乱视野 ≈ O(B)，**位移**和**邻居相关性**两个指标一起看，配合 shard 级打乱。
- **预取 + 流水重叠**让吞吐 = 1/max(各级)；先**定位瓶颈级**再优化（Amdahl 日常版）。
- 方法论：每个「有限内存的近似」都把**质量量化**、用**吞吐账**定位瓶颈。

下一站：**模块 03 · Tokenization 吞吐** —— 加载之前要先把文本变 token，万亿 token 怎么分得快、还不在 padding 上烧算力？